In [119]:
# 1. Cargar variables de entorno y crear el cliente
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic

# Ruta explicita al .env; si no existe, busca uno hacia arriba desde la carpeta actual
ENV_PATH = Path(r"C:\Users\jacob\OneDrive\Escritorio\CursoAPI_v2\.env")
loaded = load_dotenv(ENV_PATH) if ENV_PATH.exists() else load_dotenv(find_dotenv(usecwd=True))

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "No se encontro ANTHROPIC_API_KEY. Revisa que el archivo se llame exactamente "
        ".env (no .env.txt) y que contenga: ANTHROPIC_API_KEY=sk-ant-..."
    )

client = Anthropic(
    max_retries=5,
    timeout=60.0,
    default_headers={"Accept-Encoding": "gzip, deflate"},
)
model = "claude-haiku-4-5"
print("OK: .env cargado, key detectada, modelo =", model)

OK: .env cargado, key detectada, modelo = claude-haiku-4-5


In [120]:
# 2. Funciones auxiliares
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [121]:
# 3. Prueba rapida de conexion (si falla, imprime la causa raiz completa)
import traceback

try:
    print(chat([{"role": "user", "content": "Responde solo: hola"}]))
except Exception as e:
    root = e
    while root.__cause__ is not None:
        root = root.__cause__
    print("ERROR:", type(e).__name__, "->", type(root).__name__, root)
    traceback.print_exception(type(root), root, root.__traceback__)

Hola


In [122]:
# 4. Generar el dataset de evaluacion
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task"
        "format" : "python" | "json" | "regex",
        "solution_criteria" : "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [123]:
# 5. Ejecutar y guardar
dataset = generate_dataset()
print(json.dumps(dataset, indent=2, ensure_ascii=False))

with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print("Guardado en dataset.json")

[
  {
    "task": "Extract all S3 bucket names from AWS CloudTrail logs in JSON format",
    "format": "regex",
    "solution_criteria": "The regex should match and capture S3 bucket names in ARN format (arn:aws:s3:::bucket-name) and return only the bucket names without the arn prefix"
  },
  {
    "task": "Create a Python function that validates AWS IAM policy JSON and returns True if all required keys (Version, Statement) are present",
    "format": "python",
    "solution_criteria": "Function should accept a dictionary, check for 'Version' and 'Statement' keys, and return a boolean. Should handle missing keys gracefully without raising exceptions"
  },
  {
    "task": "Generate a JSON CloudFormation template snippet for an AWS Lambda function that executes every 5 minutes",
    "format": "json",
    "solution_criteria": "Valid CloudFormation JSON containing a Lambda function resource and an EventBridge rule with a 5-minute cron schedule. Must include proper resource dependencies and

In [124]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

*Respond only with Python, JSON or Regex.
*Do not include any explanations, comments, or additional text.

"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages,stop_sequences=["```"])
    return output

In [125]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [126]:
import json
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [127]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score=model_grade["score"]
    reasoning=model_grade["reasoning"]


    syntax_score = grade_syntax(output, test_case)
    score=(model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [128]:
def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = sum(result["score"] for result in results) / len(results)
    print(f"Average score across {len(results)} test cases: {average_score:.2f}")
    return results



In [129]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score across 3 test cases: 6.67


In [130]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nimport re\nfrom typing import Set\n\ndef extract_s3_buckets_from_cloudtrail(log_data: str) -> Set[str]:\n    \"\"\"Extract S3 bucket names from CloudTrail logs.\"\"\"\n    buckets = set()\n    \n    try:\n        logs = json.loads(log_data)\n    except json.JSONDecodeError:\n        return buckets\n    \n    # Handle both single log entry and array of logs\n    entries = logs if isinstance(logs, list) else [logs]\n    \n    for entry in entries:\n        if not isinstance(entry, dict):\n            continue\n            \n        # Check requestParameters for bucket name\n        if \"requestParameters\" in entry and isinstance(entry[\"requestParameters\"], dict):\n            request_params = entry[\"requestParameters\"]\n            \n            if \"bucketName\" in request_params:\n                buckets.add(request_params[\"bucketName\"])\n            \n            if \"key\" in request_params and isinstance(request_params[\"key\"], str):\n    